In [1]:
import scanpy as sc
import pandas as pd
import squidpy as sq
import matplotlib.pyplot as plt

# 1. 读取数据
adata = sc.read_h5ad('E:\\ST_Graduation_Project\\spagraph_data\\database\\GSE144240\\GSE144236_P2_ST.h5ad')
composition = pd.read_csv('E:\\ST_Graduation_Project\\spagraph_data\\evaluate\\GSE144236\\Soatuak_composition.csv', index_col=0)

# 2. 将连续的细胞比例转化为离散的主导细胞类型 (Dominant Cell Type)
# idxmax(axis=1) 会自动找出每一行中数值最大（比例最高）的列名（细胞类型）
dominant_cell_types = composition.idxmax(axis=1)

# 3. 将标签对齐并安全地合并到 adata.obs 中
# 使用 reindex 确保 csv 中的 spot 名字和 h5ad 中的严格对齐，避免顺序错乱
adata.obs['dominant_cell_type'] = dominant_cell_types.reindex(adata.obs.index)

# Squidpy 强制要求聚类的 key 必须是 pandas 的 categorical 类型
adata.obs['dominant_cell_type'] = adata.obs['dominant_cell_type'].astype('category')

# 检查一下是否成功加进去了
print("标签合并成功，前五个 spot 的主导细胞类型为：\n", adata.obs['dominant_cell_type'].head())

# ----------------- 开始 Squidpy 空间分析 -----------------

# 4. 构建空间邻域图
# 对于 10X Visium 数据，通常 spot 呈六边形网格排列，所以 n_neighs=6，coord_type 设为 "grid" 或 "generic" (如果报错可换为 generic)
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=6)

# 5. 计算邻域富集分数 (基于置换检验)
sq.gr.nhood_enrichment(adata, cluster_key="dominant_cell_type")

# 6. 可视化绘制富集热图
sq.pl.nhood_enrichment(
    adata,
    cluster_key="dominant_cell_type",
    method="ward",         # Ward 聚类算法会将喜欢聚在一起的细胞排在相邻位置
    cmap="coolwarm",       # 强烈推荐红蓝配色：红色为显著富集，蓝色为显著排斥
    figsize=(8, 8),
    title="Neighborhood Enrichment in SCC Tissue"
)

plt.show()

e:\ST_Graduation_Project\spagraph_data\.venv\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(


ModuleNotFoundError: No module named 'pkg_resources'